In [38]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [39]:
test_folder_path = Path("./tests")
test = "spectra"

In [40]:
with open(test_folder_path / test / f"{test}_specs.json" ) as f:
    test_specs = json.load(f)

norms_path = test_folder_path / test / f"{test}_norms_ita.csv"
norms = pd.read_csv(norms_path)

In [41]:
def convert_to_matrices(test_specs):
    # get test length
    test_length = test_specs["length"]
    # init matrices to be returned
    df_straight = pd.DataFrame()
    df_reversed = pd.DataFrame()
    # iterate over scales
    for scale in test_specs["scales"]:
        # get scale label, straight and reversed items
        scale_label, straight_items_indices, reversed_items_indices = scale
        # iterate over type of items (either straight or reversed)
        for df, items_indices in [(df_straight, straight_items_indices), (df_reversed, reversed_items_indices)]:
            # init new series with zeroes 
            zeros = pd.Series(np.zeros(test_length))
            # correct items indices, since items are 1-based while matrix indices are 0-based
            matrix_indices = pd.Series(items_indices).sub(1)
            # "switch to 1" scale items in zeros series
            zeros[matrix_indices] = 1
            # add scale items matrix to df
            df[scale_label] = zeros
    return df_straight, df_reversed

def compute_raw_scores(data, test_specs, score_strategy = "sum", split_results = False):
    
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    
    ############################################################
    # straight items
    ############################################################
    
    # compute sum of straight items
    score_straight_sum = np.dot(answers.fillna(0), df_straight)
    # compute mean of straight items
    with np.errstate(divide='ignore', invalid='ignore'):
        score_straight_mean = np.true_divide(score_straight_sum, df_straight.sum(axis=0).to_numpy())
        score_straight_mean[score_straight_mean == np.inf] = 0
        score_straight_mean = np.nan_to_num(score_straight_mean)

    ############################################################
    # reversed items
    ############################################################
    
    # compute amount to use for reversed items
    rev = test_specs["likert"]["min"] + test_specs["likert"]["max"]
    # compute sum of reversed items
    score_reversed_sum = np.dot(rev - answers.fillna(rev), df_reversed)
    # compute mean of reversed items
    with np.errstate(divide='ignore', invalid='ignore'):
        score_reversed_mean = np.true_divide(score_reversed_sum, df_reversed.sum(axis=0).to_numpy())
        score_reversed_mean[score_reversed_mean == np.inf] = 0
        score_reversed_mean = np.nan_to_num(score_reversed_mean)

    #############################################################
    # final results
    ############################################################

    # if we want results splitted by straight/reversed items
    if split_results:
        return score_straight_sum, score_reversed_sum
    # otherwise return combined results
    else:
        # compute final results
        results.loc[:, df_straight.columns] = score_straight_sum + score_reversed_sum
        if score_strategy == "mean":
            results.loc[:, df_straight.columns] = score_straight_mean + score_reversed_mean
        
        # return results
        return results.astype(int)

def count_items_per_scale(data, test_specs, split_results = False):
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    # merge straight and reversed items matrices
    df_tot = df_straight + df_reversed
    # if we want results splitted by straight/reversed items
    if split_results:
        return df_straight.sum(), df_reversed.sum()
    # otherwise return combined results
    return df_tot.sum()

def count_missing_items_per_scale(data, test_specs, split_results = False):
    # init results
    results = pd.DataFrame()
    # clone data
    answers = data.copy()
    # get scales items matrices (two separate matrices for straight and reversed items)
    df_straight, df_reversed = convert_to_matrices(test_specs)
    # merge straight and reversed items matrices
    df_tot = df_straight + df_reversed
    # if we want results splitted by straight/reversed items
    if split_results:
        return (
            df_straight.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum()),
            df_reversed.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum())
        )
    # otherwise return combined results
    return df_tot.apply(lambda x: answers.T.loc[x.astype(bool).values].isna().sum())

def compute_raw_scores_compensate_for_missing_items(data, test_specs):
    
    # get splitted data that is needed for computing raw scores while compensating for missing items
    score_straight, score_reversed = compute_raw_scores(data, test_specs, score_strategy = "sum", split_results = True)
    missing_straight, missing_reversed = count_missing_items_per_scale(data, test_specs, split_results = True)
    items_straight, items_reversed = count_items_per_scale(data, test_specs, split_results = True)
    # init list
    components = []
    # compute corrected raw scores 
    for raw_scores, missing_items_by_scale, items_by_scale in [
        (score_straight, missing_straight, items_straight),
        (score_reversed, missing_reversed, items_reversed)
    ]: 
        # compute how many items where effectively responded (by scale)
        items_with_answers_by_scale = items_by_scale - missing_items_by_scale
        with np.errstate(divide='ignore', invalid='ignore'):
            # compute mean responses (by scale)
            mean_results= np.true_divide(raw_scores, items_with_answers_by_scale.astype(int))
            mean_results[mean_results == np.inf] = 0
            mean_results = np.nan_to_num(mean_results)
            # compute corrected results (by scale)
            corrected_results = mean_results * items_by_scale.to_numpy().T
            components.append(corrected_results)
    # return results as a pandas DataFrame
    return pd.DataFrame(components[0] + components[1], index=data.index, columns=items_straight.index).astype(int)
    
def compute_standard_score(s, **kwargs):
    # get kwargs
    norms, type = kwargs["norms"],  kwargs["type"]
    # return standard scores
    return norms[norms["scale"].eq(s.name)].take(s)[type].values

In [42]:
# load data
data = pd.read_csv("data_spectra.csv")
data.head()

,i1,i2,i3,i4,i5,i6,i7,i8,i9,i10,i11,i12,i13,i14,i15,i16,i17,i18,i19,i20,i21,i22,i23,i24,i25,i26,i27,i28,i29,i30,i31,i32,i33,i34,i35,i36,i37,i38,i39,i40,i41,i42,i43,i44,i45,i46,i47,i48,i49,i50,i51,i52,i53,i54,i55,i56,i57,i58,i59,i60,i61,i62,i63,i64,i65,i66,i67,i68,i69,i70,i71,i72,i73,i74,i75,i76,i77,i78,i79,i80,i81,i82,i83,i84,i85,i86,i87,i88,i89,i90,i91,i92,i93,i94,i95,i96
0,4,3,2,1,3,1,1,1,2,1,1,1,2,1,1,1,1,4,3,3,2,3,1,2,1,1,1,1,3,1,1,1,5,5,2,3,1,3,1,1,1,1,1,1,1,5,1,1,5,1,5,3,3,1,4,1,1,2,1,1,1,1,5,1,2,2,3,3,3,4,1,1,2,1,1,1,3,1,1,1,3,3,4,1,2,1,1,1,2,4,1,4,3,1,2,1


In [43]:
items_by_scale = count_items_per_scale(data, test_specs)
items_by_scale

dep      6.0
anx      8.0
soc      5.0
pts      6.0
alc      6.0
agg      6.0
anti     8.0
drg      6.0
psy      6.0
par      6.0
man      5.0
gra      6.0
cog      5.0
pf       8.0
sui      6.0
int     25.0
ext     26.0
ri      23.0
gpi     74.0
dtype: float64

In [44]:
missing_items_by_scale = count_missing_items_per_scale(data, test_specs)
missing_items_by_scale

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [45]:
raw_scores = compute_raw_scores(data, test_specs)
raw_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,23,22,14,21,8,8,10,6,6,7,6,7,5,32,6,80,32,26,138


In [46]:
raw_scores_corrected = compute_raw_scores_compensate_for_missing_items(data, test_specs)
raw_scores_corrected

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,23,22,14,21,8,8,10,6,6,7,6,7,5,32,6,80,32,25,138


In [47]:
t_scores = raw_scores.apply(compute_standard_score, norms=norms, type="tscore")
t_scores

,dep,anx,soc,pts,alc,agg,anti,drg,psy,par,man,gra,cog,pf,sui,int,ext,ri,gpi
0,89,63,63,67,55,45,46,48,44,48,45,44,39,51,46,74,56,44,65


In [117]:
df = pd.read_clipboard()

In [132]:
l = df.melt(id_vars=["raw1","raw2","raw3","raw4","raw5"])
m = l["value"]
m.index = range(74,374)
m = m.dropna()
m

74      36.0
75      36.0
76      37.0
77      37.0
78      38.0
79      38.0
80      38.0
81      39.0
82      39.0
83      40.0
84      40.0
85      40.0
86      41.0
87      41.0
88      42.0
89      42.0
90      43.0
91      43.0
92      43.0
93      44.0
94      44.0
95      45.0
96      45.0
97      45.0
98      46.0
99      46.0
100     47.0
101     47.0
102     47.0
103     48.0
104     48.0
105     49.0
106     49.0
107     50.0
108     50.0
109     50.0
110     51.0
111     51.0
112     52.0
113     52.0
114     52.0
115     53.0
116     53.0
117     54.0
118     54.0
119     54.0
120     55.0
121     55.0
122     56.0
123     56.0
124     57.0
125     57.0
126     57.0
127     58.0
128     58.0
129     59.0
130     59.0
131     59.0
132     60.0
133     60.0
134     61.0
135     61.0
136     61.0
137     62.0
138     62.0
139     63.0
140     63.0
141     63.0
142     64.0
143     64.0
144     65.0
145     65.0
146     66.0
147     66.0
148     66.0
149     67.0
150     67.0

In [139]:
m = pd.concat([
    pd.Series([ 36 for i in range(0,74)], name="gpi"),
    m
]).to_frame().reset_index().rename(columns={ "index": "raw", 0: "tscore"}) # .melt(id_vars="index").loc[:, ["variable", "index", "value"]]

In [142]:
m["scale"] = "gpi"
m

,raw,tscore,scale
0,0,36.0,gpi
1,1,36.0,gpi
2,2,36.0,gpi
3,3,36.0,gpi
4,4,36.0,gpi
5,5,36.0,gpi
6,6,36.0,gpi
7,7,36.0,gpi
8,8,36.0,gpi
9,9,36.0,gpi
